# DEMO: Publishing a Dashboard

In Databricks, publishing involves one critical decision: **who's credentials run the queries?**

This is the single most important concept in this module.

In this demo, we'll:
1. Open a pre-built draft dashboard
2. Publish it with **Shared Data Permissions** (publisher credentials)
3. Understand when to use **Individual Data Permissions** (viewer credentials)
4. See the Draft vs Published behavior in action
5. Make a change and observe it doesn't affect the published version

> **Note:** Run the setup cell below to create the source data AND a draft dashboard. Then follow the UI instructions.

---

In [0]:
%run ./Setup

## The Two Credential Modes

When you click **Publish**, you must choose one of two modes. This decision determines who's data access runs the queries.

### 1. Shared Data Permissions (Publisher Credentials)

**This is the default and closest to Power BI's behavior.**

| Aspect | Detail |
| --- | --- |
| How it works | Viewers run queries using the **publisher's** data permissions |
| What viewers need | Only dashboard access — NOT access to underlying tables |
| What viewers see | SAME data regardless of their own UC permissions |
| Best for | Executive dashboards, standard reports, embedded dashboards |

**Security note:** In production, publish using a **service principal's** credentials (not your personal account). This avoids the "what happens when the publisher leaves the company" problem.

### 2. Individual Data Permissions (Viewer Credentials)

| Aspect | Detail |
| --- | --- |
| How it works | Each viewer runs queries using their **OWN** credentials |
| What viewers need | SELECT access to the underlying tables in Unity Catalog |
| What viewers see | DIFFERENT results based on their own data permissions |
| Best for | Regional dashboards, compliance data, multi-tenant views |

**Key insight:** You don't need to write security rules inside the BI tool. Unity Catalog permissions ARE the security model. If someone can SELECT the data in SQL, they see it in the dashboard. If they can't, they don't.

### Decision Guide

| Scenario | Mode | Why |
| --- | --- | --- |
| Executive dashboard (all see same data) | **Shared** | Simpler; viewers don't need data access |
| Regional managers (each sees their region) | **Individual** | UC permissions handle row filtering |
| Embedded in external portal | **Shared** (service principal) | External users can't have UC accounts |
| Compliance/audit dashboard | **Individual** | Ensures viewers only see authorized data |
| Development/testing | **Shared** | Easiest for iteration |

---

## Part 1: Open the Draft Dashboard

### Step 1: Navigate to the dashboard
1. Click **Dashboards** in the left sidebar
2. Find: **Sales Dashboard - Publishing Demo** (your username in parentheses)
3. Click to open it

### What you see:
- The dashboard opens in **Draft** mode
- You can see the widgets: two bar charts (Revenue by Region, Revenue by Category) and a line chart (Monthly Trend)
- The **Publish** button appears in the top-right corner (blue button)
- Note: this dashboard has NEVER been published — only a draft exists

### Step 2: Review the datasets
1. Click the **Data** tab (left panel)
2. You'll see 3 datasets: Revenue by Region, Monthly Revenue Trend, Revenue by Category
3. Click any dataset to see its SQL query

> **Power BI parallel:** This is like having a `.pbix` file open in Desktop. The data model (datasets) and report (canvas) are all here, but it hasn't been published to the Service yet.

---

## Part 2: Publish with Shared Data Permissions

### Step 1: Click Publish
1. Click the **Publish** button (top-right corner, blue)
2. A dialog appears asking you to choose the credential mode

### Step 2: Choose credential mode
1. Select **Share data with all viewers** (this is "Shared Data Permissions")
   - This means viewers will use YOUR data access to run queries
   - Viewers do NOT need access to the underlying `gold_sales` table
2. Click **Publish**

### What happens:
- A **published snapshot** is created — a frozen copy of the current draft
- The URL changes (or you see a "Published" indicator)
- A **companion Genie Space** is automatically created
- Viewers can now access the published version

### Step 3: View the published dashboard
1. After publishing, click **View published dashboard** (or toggle between Draft/Published)
2. Notice: this is the **viewer experience** — read-only, interactive filters work, but no editing

> **Power BI parallel:** This is the moment you click "Publish" in Desktop and it appears in the Power BI Service. The key difference: in Power BI, subsequent edits to the published report are live. In Databricks, the published version is frozen until you re-publish.

---

## Part 3: Draft vs Published — The Snapshot Model

This is where Databricks differs most from Power BI. Let's see it in action.

### Step 1: Make a change to the draft
1. Switch back to the **Draft** view (click "Draft" or "Edit")
2. Make a visible change:
   - Click the text widget at the top
   - Change the title to: "Sales Performance Dashboard **[UPDATED]**"
   - Or add a new widget (e.g., a text widget)
3. The change auto-saves in the draft

### Step 2: Check the published version
1. Switch to the **Published** view (or open in a new tab)
2. Notice: **your change is NOT there** — the published version still shows the old title

### Step 3: Understand the lifecycle

| Action | Effect |
| --- | --- |
| Edit a widget | Changes the **draft** only |
| View published version | Still shows the LAST published snapshot |
| Click **Publish** again | Creates a NEW snapshot (viewers now see updates) |
| Scheduled refresh | Refreshes the **published** version's data cache |

### Step 4: Re-publish to push changes live
1. Switch back to Draft
2. Click **Publish** again
3. The updated version is now live — check the published view

> **Power BI parallel:** Power BI doesn't have this staging model. When you edit a published report in the Service, changes are live immediately. Databricks gives you a **staging area** (draft) to test changes before pushing them live. This is safer for production dashboards.

---

## Important: Compute Access

Regardless of which credential mode you choose:

**Compute access (the SQL warehouse) is always granted by the publisher's credentials.**

This means:
- The publisher must have access to a SQL warehouse
- Viewers don't need their own warehouse permissions
- The warehouse specified at publish time runs all viewer queries

| What | Shared Mode | Individual Mode |
| --- | --- | --- |
| **Data permissions** | Publisher's | Viewer's own |
| **Compute permissions** | Publisher's | Publisher's |
| **Viewers need UC table access?** | No | Yes |
| **Viewers need warehouse access?** | No | No |

---

## Production Best Practice: Publish with a Service Principal

In production, never publish with your personal account. Use a **service principal** instead.

### Why?
- If the publisher (a person) leaves the company, their credentials are revoked
- The dashboard breaks for ALL viewers
- A service principal is a non-human identity that persists regardless of team changes

### How:
1. Create a service principal in your Databricks account
2. Grant it SELECT access to the required tables in Unity Catalog
3. Grant it access to a SQL warehouse
4. Use the service principal's credentials to publish (via API or by logging in as the SP)

> **Power BI parallel:** This is like using a "data gateway" with a service account in Power BI, rather than a personal account for dataset refresh. Same principle — decouple from individual users.

---

## Part 4: Sharing the Published Dashboard

Once published, you need to share it with viewers.

### Step 1: Click Share
1. On the published dashboard, click **Share** (top-right corner)
2. A dialog appears

### Step 2: Add users or groups
1. Type a user's email or group name
2. Choose the permission level:

| Permission | What They Can Do | Power BI Equivalent |
| --- | --- | --- |
| **Can View** | View published dashboard, interact with filters | Viewer role |
| **Can Run** | View + manually trigger data refresh | Viewer with refresh |
| **Can Edit** | Full edit access to draft + publish | Contributor role |
| **Is Owner** | Edit + manage permissions + delete | Admin role |

3. Click **Share**

### Account-Level Sharing
For stakeholders outside your workspace:
- They just need to be registered in your Databricks **account** (not workspace)
- They see ONLY the published dashboard — no sidebar, no notebooks, no SQL editor
- Perfect for executives or cross-team stakeholders

> **Power BI parallel:** Account-level sharing is like sharing via a Power BI App — a curated, view-only experience.

---

---

## Summary

| Concept | Key Takeaway |
| --- | --- |
| **Publishing** | Creates a frozen snapshot; viewers see published version |
| **Shared Data Permissions** | Publisher's credentials run queries; viewers don't need table access |
| **Individual Data Permissions** | Viewer's own credentials; UC permissions determine what they see |
| **Draft vs Published** | Edits only affect draft; must re-publish to update published version |
| **Compute** | Always uses publisher's warehouse regardless of credential mode |
| **Service Principal** | Production best practice; avoids dependency on individual users |
| **Sharing** | Can View / Can Run / Can Edit / Is Owner; account-level for external users |

### The Publishing Workflow

```
1. Build dashboard (draft)         →  Auto-saved, iterate freely
2. Publish (choose credential mode) →  Frozen snapshot created
3. Share (add users/groups)          →  Set permission levels
4. Edit draft (make changes)         →  Published version unchanged
5. Re-publish when ready             →  New snapshot goes live
```

**Key mental model shift from Power BI:**
- Power BI: Published = live (edits reflect immediately)
- Databricks: Published = snapshot (explicit re-publish required)
- This gives you a **safe staging area** to test changes before pushing to viewers.

**Next:** We'll set up scheduled refreshes and subscriptions to keep the published dashboard's data fresh.